In [1]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl
import data_collection as data
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))


Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025
week = 6

# Load play-by-play data for the chosen season
def get_weekly_scorers(season, week):
    pbp = nfl.import_pbp_data(years=[season])

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

    # Show results
scorers = get_weekly_scorers(2025, week)
scorers.head(10)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0031610,D.Waller,1
2,00-0032398,C.Moore,1
3,00-0033110,T.Higbee,1
4,00-0033280,C.McCaffrey,1
5,00-0033357,T.Hill,1
6,00-0033375,T.Patrick,1
7,00-0033873,P.Mahomes,1
8,00-0033908,C.Kupp,1
9,00-0034351,D.Goedert,1


In [3]:
predictions = pd.read_csv(f'data/predictions_week_{week}_rf.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(10)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0039139,Jahmyr Gibbs,RB,DET,0.605607,-160.0,-0.009778,0.615385
1,00-0037248,James Cook,RB,BUF,0.594122,-155.0,-0.013721,0.607843
2,00-0036223,Jonathan Taylor,RB,IND,0.592335,-265.0,-0.133692,0.726027
3,00-0039075,Puka Nacua,WR,LA,0.578073,-120.0,0.032619,0.545455
4,00-0036997,Javonte Williams,RB,DAL,0.566890,-210.0,-0.110529,0.677419
5,00-0036158,J.K. Dobbins,RB,DEN,0.565085,-125.0,0.009529,0.555556
6,00-0035700,Josh Jacobs,RB,GB,0.560620,-265.0,-0.165407,0.726027
7,00-0037840,Kyren Williams,RB,LA,0.559859,-175.0,-0.076505,0.636364
8,00-0038542,Bijan Robinson,RB,ATL,0.530860,-170.0,-0.098770,0.629630
9,00-0039040,De'Von Achane,RB,MIA,0.511112,-115.0,-0.023772,0.534884


In [4]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False]).head(10)

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob,tds
0,00-0036223,Jonathan Taylor,RB,IND,0.592335,-265.0,-0.133692,0.726027,1
1,00-0035700,Josh Jacobs,RB,GB,0.560620,-265.0,-0.165407,0.726027,2
2,00-0037840,Kyren Williams,RB,LA,0.559859,-175.0,-0.076505,0.636364,1
3,00-0038542,Bijan Robinson,RB,ATL,0.530860,-170.0,-0.098770,0.629630,1
4,00-0039040,De'Von Achane,RB,MIA,0.511112,-115.0,-0.023772,0.534884,2
5,00-0040122,Ashton Jeanty,RB,LV,0.467337,-150.0,-0.132663,0.600000,1
6,00-0033280,Christian McCaffrey,RB,SF,0.442610,-175.0,-0.193754,0.636364,1
7,00-0037247,George Pickens,WR,DAL,0.435447,115.0,-0.029670,0.465116,1
8,00-0036275,D'Andre Swift,RB,CHI,0.414750,120.0,-0.039796,0.454545,1
9,00-0036139,Rico Dowdle,RB,CAR,0.389501,-120.0,-0.155954,0.545455,1


In [5]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [6]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[round(predictions['model_edge'], 2) >= 0.05] 
ev = ev[ev['price'] <= 400]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
27,00-0036912,DeVonta Smith,WR,PHI,0.392427,250.0,0.106713,0.285714
10,00-0034348,Courtland Sutton,WR,DEN,0.498778,140.0,0.082111,0.416667


In [7]:
simulate_betting(ev, scorers)

{'bets': 2, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -20.0, 'roi': -1.0}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034348,Courtland Sutton,DEN,WR,140.0,0.498778,0.082111,0,False,-10.0,0.0
1,00-0036912,DeVonta Smith,PHI,WR,250.0,0.392427,0.106713,0,False,-10.0,0.0


In [8]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0039139,Jahmyr Gibbs,RB,DET,0.605607,-160.0,-0.009778,0.615385
1,00-0037248,James Cook,RB,BUF,0.594122,-155.0,-0.013721,0.607843
2,00-0036223,Jonathan Taylor,RB,IND,0.592335,-265.0,-0.133692,0.726027
4,00-0036997,Javonte Williams,RB,DAL,0.566890,-210.0,-0.110529,0.677419
5,00-0036158,J.K. Dobbins,RB,DEN,0.565085,-125.0,0.009529,0.555556
6,00-0035700,Josh Jacobs,RB,GB,0.560620,-265.0,-0.165407,0.726027
7,00-0037840,Kyren Williams,RB,LA,0.559859,-175.0,-0.076505,0.636364
8,00-0038542,Bijan Robinson,RB,ATL,0.530860,-170.0,-0.098770,0.629630
9,00-0039040,De'Von Achane,RB,MIA,0.511112,-115.0,-0.023772,0.534884
11,00-0034844,Saquon Barkley,RB,PHI,0.494728,-160.0,-0.120657,0.615385


In [9]:
simulate_betting(top_rb, scorers)


{'bets': 10, 'hits': 5, 'hit_rate': 0.5, 'total_profit': -22.16, 'roi': -0.222}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039139,Jahmyr Gibbs,DET,RB,-160.0,0.605607,-0.009778,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-155.0,0.594122,-0.013721,0,False,-10.000000,0.000000
2,00-0036223,Jonathan Taylor,IND,RB,-265.0,0.592335,-0.133692,1,True,3.773585,13.773585
3,00-0036997,Javonte Williams,DAL,RB,-210.0,0.566890,-0.110529,0,False,-10.000000,0.000000
4,00-0036158,J.K. Dobbins,DEN,RB,-125.0,0.565085,0.009529,0,False,-10.000000,0.000000
5,00-0035700,Josh Jacobs,GB,RB,-265.0,0.560620,-0.165407,2,True,3.773585,13.773585
6,00-0037840,Kyren Williams,LA,RB,-175.0,0.559859,-0.076505,1,True,5.714286,15.714286
7,00-0038542,Bijan Robinson,ATL,RB,-170.0,0.530860,-0.098770,1,True,5.882353,15.882353
8,00-0039040,De'Von Achane,MIA,RB,-115.0,0.511112,-0.023772,2,True,8.695652,18.695652
9,00-0034844,Saquon Barkley,PHI,RB,-160.0,0.494728,-0.120657,0,False,-10.000000,0.000000


In [10]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)


{'bets': 10, 'hits': 3, 'hit_rate': 0.3, 'total_profit': -32.0, 'roi': -0.32}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.0,0.0
1,00-0034348,Courtland Sutton,DEN,WR,140.0,0.498778,0.082111,0,False,-10.0,0.0
2,00-0040129,Emeka Egbuka,TB,WR,100.0,0.475819,-0.024181,0,False,-10.0,0.0
3,00-0031381,Davante Adams,LA,WR,100.0,0.465665,-0.034335,0,False,-10.0,0.0
4,00-0037247,George Pickens,DAL,WR,115.0,0.435447,-0.029670,1,True,11.5,21.5
5,00-0035719,Deebo Samuel Sr.,WAS,WR,120.0,0.433162,-0.021383,0,False,-10.0,0.0
6,00-0036963,Amon-Ra St. Brown,DET,WR,110.0,0.431889,-0.044302,0,False,-10.0,0.0
7,00-0036912,DeVonta Smith,PHI,WR,250.0,0.392427,0.106713,0,False,-10.0,0.0
8,00-0038543,Jaxon Smith-Njigba,SEA,WR,125.0,0.381401,-0.063044,1,True,12.5,22.5
9,00-0037238,Drake London,ATL,WR,140.0,0.369188,-0.047479,1,True,14.0,24.0


In [11]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 59.5, 'roi': 1.19}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0038996,Tucker Kraft,GB,TE,175.0,0.331078,-0.032558,1,True,17.5,27.5
1,00-0040128,Tyler Warren,IND,TE,155.0,0.330150,-0.062007,1,True,15.5,25.5
2,00-0039065,Sam LaPorta,DET,TE,220.0,0.307752,-0.004748,1,True,22.0,32.0
3,00-0038933,Dalton Kincaid,BUF,TE,225.0,0.296366,-0.011327,0,False,-10.0,0.0
4,00-0038041,Jake Ferguson,DAL,TE,145.0,0.287213,-0.120950,1,True,14.5,24.5


In [12]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 3.4, 'roi': 0.068}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-105.0,0.257904,-0.254291,0,False,-10.000000,0.000000
1,00-0036945,Justin Fields,NYJ,QB,240.0,0.239585,-0.054533,0,False,-10.000000,0.000000
2,00-0036389,Jalen Hurts,PHI,QB,-145.0,0.239096,-0.352741,1,True,6.896552,16.896552
3,00-0033873,Patrick Mahomes,KC,QB,265.0,0.225728,-0.048245,1,True,26.500000,36.500000
4,00-0036355,Justin Herbert,LAC,QB,400.0,0.199050,-0.000950,0,False,-10.000000,0.000000


In [13]:
ev_wr = predictions[predictions['model_edge'] >= 0.00]
ev_wr = ev_wr[ev_wr['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
simulate_betting(ev_wr, scorers)


{'bets': 10, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -100.0, 'roi': -1.0}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.0,0.0
1,00-0034348,Courtland Sutton,DEN,WR,140.0,0.498778,0.082111,0,False,-10.0,0.0
2,00-0036912,DeVonta Smith,PHI,WR,250.0,0.392427,0.106713,0,False,-10.0,0.0
3,00-0035676,A.J. Brown,PHI,WR,195.0,0.349840,0.010857,0,False,-10.0,0.0
4,00-0037664,Alec Pierce,IND,WR,330.0,0.270440,0.037882,0,False,-10.0,0.0
5,00-0038619,Andrei Iosivas,CIN,WR,550.0,0.169799,0.015953,0,False,-10.0,0.0
6,00-0031236,Brandin Cooks,NO,WR,650.0,0.144984,0.011650,0,False,-10.0,0.0
7,00-0039424,Devaughn Vele,NO,WR,1500.0,0.129202,0.066702,0,False,-10.0,0.0
8,00-0038465,Malik Heath,GB,WR,750.0,0.127960,0.010313,0,False,-10.0,0.0
9,00-0030564,DeAndre Hopkins,BAL,WR,800.0,0.127098,0.015987,0,False,-10.0,0.0


In [14]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 10, 'hits': 5, 'hit_rate': 0.5, 'total_profit': -22.16, 'roi': -0.222}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039139,Jahmyr Gibbs,DET,RB,-160.0,0.605607,-0.009778,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-155.0,0.594122,-0.013721,0,False,-10.000000,0.000000
2,00-0036223,Jonathan Taylor,IND,RB,-265.0,0.592335,-0.133692,1,True,3.773585,13.773585
3,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.000000,0.000000
4,00-0036997,Javonte Williams,DAL,RB,-210.0,0.566890,-0.110529,0,False,-10.000000,0.000000
5,00-0036158,J.K. Dobbins,DEN,RB,-125.0,0.565085,0.009529,0,False,-10.000000,0.000000
6,00-0035700,Josh Jacobs,GB,RB,-265.0,0.560620,-0.165407,2,True,3.773585,13.773585
7,00-0037840,Kyren Williams,LA,RB,-175.0,0.559859,-0.076505,1,True,5.714286,15.714286
8,00-0038542,Bijan Robinson,ATL,RB,-170.0,0.530860,-0.098770,1,True,5.882353,15.882353
9,00-0039040,De'Von Achane,MIA,RB,-115.0,0.511112,-0.023772,2,True,8.695652,18.695652


In [15]:
top_vegas = predictions.sort_values('market_implied_prob', ascending=False).head(19)
simulate_betting(top_vegas, scorers)

{'bets': 19,
 'hits': 10,
 'hit_rate': 0.526,
 'total_profit': -26.55,
 'roi': -0.14}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039139,Jahmyr Gibbs,DET,RB,-160.0,0.605607,-0.009778,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-155.0,0.594122,-0.013721,0,False,-10.000000,0.000000
2,00-0036223,Jonathan Taylor,IND,RB,-265.0,0.592335,-0.133692,1,True,3.773585,13.773585
3,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.000000,0.000000
4,00-0036997,Javonte Williams,DAL,RB,-210.0,0.566890,-0.110529,0,False,-10.000000,0.000000
5,00-0036158,J.K. Dobbins,DEN,RB,-125.0,0.565085,0.009529,0,False,-10.000000,0.000000
6,00-0035700,Josh Jacobs,GB,RB,-265.0,0.560620,-0.165407,2,True,3.773585,13.773585
7,00-0037840,Kyren Williams,LA,RB,-175.0,0.559859,-0.076505,1,True,5.714286,15.714286
8,00-0038542,Bijan Robinson,ATL,RB,-170.0,0.530860,-0.098770,1,True,5.882353,15.882353
9,00-0039040,De'Von Achane,MIA,RB,-115.0,0.511112,-0.023772,2,True,8.695652,18.695652


In [16]:
vegas_similar = predictions[predictions['model_edge'] < 0.05]
vegas_similar = vegas_similar[vegas_similar['model_edge'] > 0]
vegas_similar = vegas_similar[vegas_similar['price'] < 300]
simulate_betting(vegas_similar, scorers)

{'bets': 6, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -60.0, 'roi': -1.0}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.0,0.0
1,00-0036158,J.K. Dobbins,DEN,RB,-125.0,0.565085,0.009529,0,False,-10.0,0.0
2,00-0035676,A.J. Brown,PHI,WR,195.0,0.349840,0.010857,0,False,-10.0,0.0
3,00-0036919,Kenneth Gainwell,PIT,RB,230.0,0.328904,0.025874,0,False,-10.0,0.0
4,00-0039032,Tyjae Spears,TEN,RB,295.0,0.293691,0.040526,0,False,-10.0,0.0
5,00-0037197,Isiah Pacheco,KC,RB,260.0,0.284324,0.006547,0,False,-10.0,0.0


In [17]:
top_25 = predictions.head(25)
simulate_betting(top_25, scorers)

{'bets': 25,
 'hits': 9,
 'hit_rate': 0.36,
 'total_profit': -96.28,
 'roi': -0.385}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039139,Jahmyr Gibbs,DET,RB,-160.0,0.605607,-0.009778,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-155.0,0.594122,-0.013721,0,False,-10.000000,0.000000
2,00-0036223,Jonathan Taylor,IND,RB,-265.0,0.592335,-0.133692,1,True,3.773585,13.773585
3,00-0039075,Puka Nacua,LA,WR,-120.0,0.578073,0.032619,0,False,-10.000000,0.000000
4,00-0036997,Javonte Williams,DAL,RB,-210.0,0.566890,-0.110529,0,False,-10.000000,0.000000
5,00-0036158,J.K. Dobbins,DEN,RB,-125.0,0.565085,0.009529,0,False,-10.000000,0.000000
6,00-0035700,Josh Jacobs,GB,RB,-265.0,0.560620,-0.165407,2,True,3.773585,13.773585
7,00-0037840,Kyren Williams,LA,RB,-175.0,0.559859,-0.076505,1,True,5.714286,15.714286
8,00-0038542,Bijan Robinson,ATL,RB,-170.0,0.530860,-0.098770,1,True,5.882353,15.882353
9,00-0039040,De'Von Achane,MIA,RB,-115.0,0.511112,-0.023772,2,True,8.695652,18.695652


In [31]:
import pandas as pd
from typing import Optional


def append_week_lines_to_historic(
    week_lines_csv: str = "data/week_6_lines.csv",
    historic_csv: str = "data/historic_lines.csv",
    teams_csv: str = "nfl_teams.csv",
    schedule_season: int = 2025,
    schedule_week: int = 6,
    bookmaker_preference: Optional[str] = "DraftKings"
) -> int:
    """Append weekly spread lines into historic_lines.csv, preserving schema.

    - Reads week-level lines in the format of week_2_lines.csv (two rows/team per game).
    - Maps full team names to IDs matching historic_lines.csv via nfl_teams.csv.
    - Aggregates to one row per game: picks the favorite (negative point), keeps total.
    - Appends new rows to historic_lines.csv with the same columns and blank index header.

    Returns
    -------
    int
        Number of rows appended (deduped against existing season/week/home/away).
    """
    # Load team mapping (full name -> team_id used in historic file)
    teams_df = pd.read_csv(teams_csv)
    name_to_id = dict(zip(teams_df["team_name"].astype(str), teams_df["team_id"].astype(str)))

    # Load week lines and filter to spreads (and bookmaker if provided)
    week_df = pd.read_csv(week_lines_csv)
    if "market" in week_df.columns:
        week_df = week_df[week_df["market"].str.lower() == "spreads"].copy()
    if bookmaker_preference and "bookmaker" in week_df.columns:
        week_df = week_df[week_df["bookmaker"].astype(str) == bookmaker_preference].copy()

    # Normalize numeric fields
    if "point" in week_df.columns:
        week_df["point"] = pd.to_numeric(week_df["point"], errors="coerce")
    # 'over/under' has a slash in the name; keep safe access
    ou_col = "over/under" if "over/under" in week_df.columns else (
        "over_under" if "over_under" in week_df.columns else None
    )
    if ou_col is not None:
        week_df[ou_col] = pd.to_numeric(week_df[ou_col], errors="coerce")

    # Group to one row per game
    required_cols = {"game_id", "home_team", "away_team", "label"}
    missing = [c for c in required_cols if c not in week_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {week_lines_csv}: {missing}")

    records = []
    for game_id, grp in week_df.groupby("game_id", sort=False):
        home_team_name = str(grp["home_team"].iloc[0])
        away_team_name = str(grp["away_team"].iloc[0])

        # Determine favorite: row with the most negative spread (minimum point)
        grp_nonnull = grp.dropna(subset=["point"]) if "point" in grp.columns else grp.copy()
        if grp_nonnull.empty:
            # If we can't determine a favorite, skip this game
            continue
        fav_idx = grp_nonnull["point"].idxmin()
        fav_team_name = str(grp_nonnull.loc[fav_idx, "label"])  # team name in the bet label
        spread_favorite = float(grp_nonnull.loc[fav_idx, "point"])  # should be negative

        # Over/Under line: take first non-null within the game
        if ou_col is not None:
            ou_series = grp_nonnull[ou_col].dropna()
            over_under_line = float(ou_series.iloc[0]) if not ou_series.empty else None
        else:
            over_under_line = None

        # Map to team IDs used in historic file
        home_id = name_to_id.get(home_team_name)
        away_id = name_to_id.get(away_team_name)
        fav_id = name_to_id.get(fav_team_name)

        if home_id is None or away_id is None or fav_id is None:
            # Try a couple of common aliases
            alias = {
                "LA Rams": "Los Angeles Rams",
                "LA Chargers": "Los Angeles Chargers",
                "LV Raiders": "Las Vegas Raiders",
                "Washington": "Washington Commanders",
            }
            home_id = home_id or name_to_id.get(alias.get(home_team_name, home_team_name))
            away_id = away_id or name_to_id.get(alias.get(away_team_name, away_team_name))
            fav_id = fav_id or name_to_id.get(alias.get(fav_team_name, fav_team_name))

        if home_id is None or away_id is None or fav_id is None:
            raise KeyError(
                f"Missing team_id mapping. home='{home_team_name}'->{home_id}, "
                f"away='{away_team_name}'->{away_id}, favorite='{fav_team_name}'->{fav_id}"
            )

        records.append({
            "schedule_season": int(schedule_season),
            "schedule_week": int(schedule_week),
            "team_home": home_team_name,
            "team_away": away_team_name,
            "team_favorite_id": fav_id,
            "spread_favorite": spread_favorite,
            "over_under_line": over_under_line,
            "schedule_playoff": False,
            "team_home_id": home_id,
            "team_away_id": away_id,
        })

    new_rows_df = pd.DataFrame.from_records(records)
    if new_rows_df.empty:
        return 0

    # Load historic lines with existing index (blank header) and dedupe by key
    historic_df = pd.read_csv(historic_csv, index_col=0, low_memory=False)
    historic_df.index.name = ""  # Ensure blank header on index when saving

    new_rows_df["__key"] = (
        new_rows_df["schedule_season"].astype(str)
        + "|" + new_rows_df["schedule_week"].astype(str)
        + "|" + new_rows_df["team_home"].astype(str)
        + "|" + new_rows_df["team_away"].astype(str)
    )
    hist_keys = set(
        (historic_df["schedule_season"].astype(str)
         + "|" + historic_df["schedule_week"].astype(str)
         + "|" + historic_df["team_home"].astype(str)
         + "|" + historic_df["team_away"].astype(str))
        .values
    )

    new_rows_df = new_rows_df[~new_rows_df["__key"].isin(hist_keys)].drop(columns=["__key"])  # anti-join
    if new_rows_df.empty:
        return 0

    # Assign sequential index values continuing from existing max index
    try:
        start_index = int(pd.to_numeric(pd.Series(historic_df.index)).max())
    except Exception:
        # If index isn't numeric for some reason, fall back to length-1
        start_index = len(historic_df) - 1

    new_index = list(range(start_index + 1, start_index + 1 + len(new_rows_df)))
    new_rows_df.index = new_index
    new_rows_df.index.name = ""  # keep blank index header

    # Concatenate and persist
    out_df = pd.concat([historic_df, new_rows_df], axis=0)
    out_df.index.name = ""
    out_df.to_csv(historic_csv)

    return len(new_rows_df)

# Example usage (uncomment to run):
# appended = append_week_lines_to_historic()
# print(f"Appended {appended} rows to historic_lines.csv")



In [32]:
appended = append_week_lines_to_historic()

In [33]:
week_current = pd.read_csv('data/predictions_week_6_rf.csv')
week_current = week_current[['player_id', 'player_display_name', 'position', 'team', 'opponent_team', 'predicted_touchdown_probability', 'price', 'model_edge', 'market_implied_prob']]
top_rb = week_current[week_current['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = week_current[week_current['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = week_current[week_current['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = week_current[week_current['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
ev = week_current[week_current['model_edge'] >= 0.00]
ev = ev[ev['price'] <= 400]
ev = ev.sort_values('model_edge', ascending=False)
high_prob = week_current[week_current['predicted_touchdown_probability'] >= 0.50]








In [34]:
top_rb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0039139,Jahmyr Gibbs,RB,DET,KC,0.605607,-160.0,-0.009778,0.615385
1,00-0037248,James Cook,RB,BUF,ATL,0.594122,-155.0,-0.013721,0.607843
2,00-0036223,Jonathan Taylor,RB,IND,ARI,0.592335,-265.0,-0.133692,0.726027
4,00-0036997,Javonte Williams,RB,DAL,CAR,0.566890,-210.0,-0.110529,0.677419
5,00-0036158,J.K. Dobbins,RB,DEN,NYJ,0.565085,-125.0,0.009529,0.555556
6,00-0035700,Josh Jacobs,RB,GB,CIN,0.560620,-265.0,-0.165407,0.726027
7,00-0037840,Kyren Williams,RB,LA,BAL,0.559859,-175.0,-0.076505,0.636364
8,00-0038542,Bijan Robinson,RB,ATL,BUF,0.530860,-170.0,-0.098770,0.629630
9,00-0039040,De'Von Achane,RB,MIA,LAC,0.511112,-115.0,-0.023772,0.534884
11,00-0034844,Saquon Barkley,RB,PHI,NYG,0.494728,-160.0,-0.120657,0.615385


In [35]:
top_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
3,00-0039075,Puka Nacua,WR,LA,BAL,0.578073,-120.0,0.032619,0.545455
10,00-0034348,Courtland Sutton,WR,DEN,NYJ,0.498778,140.0,0.082111,0.416667
12,00-0040129,Emeka Egbuka,WR,TB,SF,0.475819,100.0,-0.024181,0.500000
14,00-0031381,Davante Adams,WR,LA,BAL,0.465665,100.0,-0.034335,0.500000
17,00-0037247,George Pickens,WR,DAL,CAR,0.435447,115.0,-0.029670,0.465116
18,00-0035719,Deebo Samuel Sr.,WR,WAS,CHI,0.433162,120.0,-0.021383,0.454545
19,00-0036963,Amon-Ra St. Brown,WR,DET,KC,0.431889,110.0,-0.044302,0.476190
27,00-0036912,DeVonta Smith,WR,PHI,NYG,0.392427,250.0,0.106713,0.285714
30,00-0038543,Jaxon Smith-Njigba,WR,SEA,JAX,0.381401,125.0,-0.063044,0.444444
33,00-0037238,Drake London,WR,ATL,BUF,0.369188,140.0,-0.047479,0.416667


In [36]:
top_te






,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
48,00-0038996,Tucker Kraft,TE,GB,CIN,0.331078,175.0,-0.032558,0.363636
50,00-0040128,Tyler Warren,TE,IND,ARI,0.330150,155.0,-0.062007,0.392157
54,00-0039065,Sam LaPorta,TE,DET,KC,0.307752,220.0,-0.004748,0.312500
59,00-0038933,Dalton Kincaid,TE,BUF,ATL,0.296366,225.0,-0.011327,0.307692
62,00-0038041,Jake Ferguson,TE,DAL,CAR,0.287213,145.0,-0.120950,0.408163


In [37]:
top_qb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
73,00-0034857,Josh Allen,QB,BUF,ATL,0.257904,-105.0,-0.254291,0.512195
81,00-0036945,Justin Fields,QB,NYJ,DEN,0.239585,240.0,-0.054533,0.294118
82,00-0036389,Jalen Hurts,QB,PHI,NYG,0.239096,-145.0,-0.352741,0.591837
91,00-0033873,Patrick Mahomes,QB,KC,DET,0.225728,265.0,-0.048245,0.273973
104,00-0036355,Justin Herbert,QB,LAC,MIA,0.199050,400.0,-0.000950,0.200000


In [38]:
ev

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
27,00-0036912,DeVonta Smith,WR,PHI,NYG,0.392427,250.0,0.106713,0.285714
10,00-0034348,Courtland Sutton,WR,DEN,NYJ,0.498778,140.0,0.082111,0.416667
60,00-0039032,Tyjae Spears,RB,TEN,LV,0.293691,295.0,0.040526,0.253165
69,00-0037664,Alec Pierce,WR,IND,ARI,0.270440,330.0,0.037882,0.232558
3,00-0039075,Puka Nacua,WR,LA,BAL,0.578073,-120.0,0.032619,0.545455
51,00-0036919,Kenneth Gainwell,RB,PIT,CLE,0.328904,230.0,0.025874,0.303030
43,00-0035676,A.J. Brown,WR,PHI,NYG,0.349840,195.0,0.010857,0.338983
5,00-0036158,J.K. Dobbins,RB,DEN,NYJ,0.565085,-125.0,0.009529,0.555556
63,00-0037197,Isiah Pacheco,RB,KC,DET,0.284324,260.0,0.006547,0.277778


In [39]:
high_prob

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0039139,Jahmyr Gibbs,RB,DET,KC,0.605607,-160.0,-0.009778,0.615385
1,00-0037248,James Cook,RB,BUF,ATL,0.594122,-155.0,-0.013721,0.607843
2,00-0036223,Jonathan Taylor,RB,IND,ARI,0.592335,-265.0,-0.133692,0.726027
3,00-0039075,Puka Nacua,WR,LA,BAL,0.578073,-120.0,0.032619,0.545455
4,00-0036997,Javonte Williams,RB,DAL,CAR,0.566890,-210.0,-0.110529,0.677419
5,00-0036158,J.K. Dobbins,RB,DEN,NYJ,0.565085,-125.0,0.009529,0.555556
6,00-0035700,Josh Jacobs,RB,GB,CIN,0.560620,-265.0,-0.165407,0.726027
7,00-0037840,Kyren Williams,RB,LA,BAL,0.559859,-175.0,-0.076505,0.636364
8,00-0038542,Bijan Robinson,RB,ATL,BUF,0.530860,-170.0,-0.098770,0.629630
9,00-0039040,De'Von Achane,RB,MIA,LAC,0.511112,-115.0,-0.023772,0.534884


In [40]:
ev_wr = week_current[week_current['model_edge'] >= 0.00]
ev_wr = ev_wr[ev_wr['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
ev_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
3,00-0039075,Puka Nacua,WR,LA,BAL,0.578073,-120.0,0.032619,0.545455
10,00-0034348,Courtland Sutton,WR,DEN,NYJ,0.498778,140.0,0.082111,0.416667
27,00-0036912,DeVonta Smith,WR,PHI,NYG,0.392427,250.0,0.106713,0.285714
43,00-0035676,A.J. Brown,WR,PHI,NYG,0.349840,195.0,0.010857,0.338983
69,00-0037664,Alec Pierce,WR,IND,ARI,0.270440,330.0,0.037882,0.232558
138,00-0038619,Andrei Iosivas,WR,CIN,GB,0.169799,550.0,0.015953,0.153846
159,00-0031236,Brandin Cooks,WR,NO,NE,0.144984,650.0,0.011650,0.133333
186,00-0039424,Devaughn Vele,WR,NO,NE,0.129202,1500.0,0.066702,0.062500
189,00-0038465,Malik Heath,WR,GB,CIN,0.127960,750.0,0.010313,0.117647
191,00-0030564,DeAndre Hopkins,WR,BAL,LA,0.127098,800.0,0.015987,0.111111


In [41]:
opp_def = pd.read_csv('data/predictions_week_5.csv')
opp_def = opp_def.groupby('opponent_team')['rushing_tds_allowed_to_RB'].mean()
opp_def = opp_def.sort_values(ascending=False)
opp_def





opponent_team
TEN    1.438464
NYG    1.147448
MIN    1.110843
WAS    1.026243
BAL    1.006525
NO     0.959442
CAR    0.956869
DAL    0.925407
DET    0.872188
LV     0.860473
BUF    0.831837
NYJ    0.800888
KC     0.795109
HOU    0.778499
CIN    0.725385
SF     0.702235
CLE    0.612509
IND    0.574584
ARI    0.567479
NE     0.532934
TB     0.497552
DEN    0.291701
LAC    0.266766
MIA    0.247480
PHI    0.240899
JAX    0.219034
SEA    0.037167
LA     0.028826
Name: rushing_tds_allowed_to_RB, dtype: float64

In [42]:
%pip install -q nflreadpy 

import nflreadpy as nfl

Note: you may need to restart the kernel to use updated packages.


In [43]:
import data_collection as data
import pandas as pd
all_years_to_load = range(2020, 2026)
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))
#nfl_df = data.get_all_historic_data(all_years_to_load, team_map)
